# 01 – Data Check (Baselines / Evaluation – Sophia)

Goal of this notebook: load one patient, check sizes and value ranges, and look at the images
before building the FBP baseline and the metrics.

**Data used here (current repo state):**
- `Original_images/patient_X/image_N.jpg` – scanner reconstructions (only patients 0–2 available)
- `Sparse_images/patient_X/slice_NN.png` – sparse-view FBP reconstructions (patients 0–8)

> Later these will be replaced by float `.npy` arrays (full-view FBP = ground truth, sparse FBP per view count).
> Only the loader functions in section 2 need to change then.

## 0. Setup

In [ ]:
# In Google Colab: clone the group repo (skipped automatically when running locally)
import os, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and not os.path.exists("/content/IPPR_CT_Reconstruction"):
    !git clone -q https://github.com/gs0232/IPPR_CT_Reconstruction.git /content/IPPR_CT_Reconstruction

REPO = "/content/IPPR_CT_Reconstruction" if IN_COLAB else "."   # local: run from the repo root
print("Repo path:", os.path.abspath(REPO))

In [ ]:
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

ORIG_DIR   = Path(REPO) / "Original_images"
SPARSE_DIR = Path(REPO) / "Sparse_images"

## 1. Which patients and slices do we have?

In [ ]:
def slice_index(path):
    # 'image_12.jpg' / 'slice_07.png' -> 12 / 7  (sort numerically, not alphabetically!)
    return int(re.search(r"(\d+)", path.stem).group(1))

def list_slices(folder, pattern):
    return sorted(folder.glob(pattern), key=slice_index)

rows = []
for d in sorted(SPARSE_DIR.glob("patient_*"), key=lambda p: int(p.name.split("_")[1])):
    pid = d.name
    n_sparse = len(list_slices(d, "*.png"))
    n_orig   = len(list_slices(ORIG_DIR / pid, "*.jpg")) if (ORIG_DIR / pid).exists() else 0
    rows.append({"patient": pid, "n_original": n_orig, "n_sparse": n_sparse,
                 "slice_counts_match": n_orig == n_sparse})
overview = pd.DataFrame(rows)
overview

## 2. Loader functions

In [ ]:
def to_gray(arr):
    # RGB(A) uint8 -> 2D float array in [0, 1]. Checks first that R, G, B are identical (pure greyscale).
    if arr.ndim == 3:
        rgb = arr[..., :3].astype(np.int16)
        max_diff = np.abs(rgb - rgb[..., :1]).max()
        if max_diff > 2:   # JPG compression can cause tiny differences
            print(f"  warning: colour channels differ by up to {max_diff}")
        arr = arr[..., 0]
    return arr.astype(np.float32) / 255.0

def load_patient(folder, pattern):
    # Returns a 3D stack (slices, H, W) as float32 in [0, 1]
    files = list_slices(folder, pattern)
    return np.stack([to_gray(np.array(Image.open(f))) for f in files])

PATIENT = "patient_0"
orig   = load_patient(ORIG_DIR / PATIENT, "*.jpg")
sparse = load_patient(SPARSE_DIR / PATIENT, "*.png")
print("original:", orig.shape, orig.dtype)
print("sparse:  ", sparse.shape, sparse.dtype)

## 3. Sizes and value ranges

In [ ]:
# Raw file properties (mode = RGB / RGBA / L, bit depth)
for name, f in [("original", list_slices(ORIG_DIR / PATIENT, "*.jpg")[0]),
                ("sparse",   list_slices(SPARSE_DIR / PATIENT, "*.png")[0])]:
    im = Image.open(f)
    a = np.array(im)
    print(f"{name:8s} {f.name:14s} mode={im.mode:5s} shape={a.shape} dtype={a.dtype}")
    if im.mode == "RGBA":
        print(f"         alpha channel unique values: {np.unique(a[..., 3])[:5]}")

In [ ]:
def stats(stack, name):
    return {"data": name, "shape": stack.shape, "min": stack.min(), "max": stack.max(),
            "mean": stack.mean().round(3), "std": stack.std().round(3),
            "unique_grey_levels": len(np.unique(stack))}

pd.DataFrame([stats(orig, "original"), stats(sparse, "sparse")])

## 4. Look at the images

In [ ]:
idx = [0, len(sparse) // 2, len(sparse) - 1]   # first, middle, last slice
fig, ax = plt.subplots(2, len(idx), figsize=(4 * len(idx), 8))
for j, i in enumerate(idx):
    ax[0, j].imshow(orig[i],   cmap="gray"); ax[0, j].set_title(f"original – slice {i}")
    ax[1, j].imshow(sparse[i], cmap="gray"); ax[1, j].set_title(f"sparse FBP – slice {i}")
for a in ax.ravel(): a.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
# Intensity histograms (background pixels dominate, so log scale on y)
fig, ax = plt.subplots(1, 2, figsize=(10, 3))
ax[0].hist(orig.ravel(), bins=256);   ax[0].set_title("original");   ax[0].set_yscale("log")
ax[1].hist(sparse.ravel(), bins=256); ax[1].set_title("sparse FBP"); ax[1].set_yscale("log")
for a in ax: a.set_xlabel("intensity [0–1]")
plt.tight_layout(); plt.show()

## 5. How much of the image is head vs. background?

In [ ]:
# Relevant for the metrics: a large black background inflates PSNR/SSIM.
THRESH = 0.05
for name, s in [("original", orig), ("sparse", sparse)]:
    frac = (s > THRESH).mean()
    print(f"{name:8s}: {100 * frac:.1f}% of pixels above {THRESH}")

## Findings / notes for the log
- Original and sparse images are **not pixel-aligned** (512×512 vs 888×888, different grey scale),
  so metrics cannot be computed between them directly.
- Plan: use **full-view FBP (984 views)** of the same sinogram as ground truth, saved as float `.npy`.
- To do: …